# Voice pipeline on qwen3.5:4b: ASR → LLM → TTS, every known optimization

**Claim:** with prewarm and every known optimization, qwen3.5:4b gives warm median
time-to-first-audio under 1500 ms (from end of speech) and answers at least 5/6
`fact-*` items correctly.

**Falsifier:** either condition fails on either LLM path. For the Ollama vs mlx-lm A/B,
no winner is named when the TTFT gap is within the per-item spread.

- Clock starts at end of speech. Endpointing (1.5 s in the live agents) is not included.
- Time to first audio = first sentence synthesized and trimmed. Speaker latency is not
  included.
- Helpers live in `pipeline.py`. Raw records go to `results.json`.

In [1]:
import time, json, platform, resource
import pipeline as p

ROUNDS = 3            # warm repeats per path; one append-only conversation per round
items = p.load_items()
results = {
    "meta": {
        "machine": platform.machine(), "python": platform.python_version(),
        "ollama_model": p.OLLAMA_MODEL, "mlx_model": p.MLX_MODEL,
        "max_tokens": p.MAX_TOKENS, "num_ctx": p.NUM_CTX,
        "system_prompt": p.SYSTEM_PROMPT, "rounds": ROUNDS,
        "items": [i["id"] for i in items],
        "optimizations": p.OPTIMIZATIONS,
    },
}
[i["id"] for i in items]

['fact-001',
 'fact-002',
 'fact-003',
 'fact-004',
 'fact-005',
 'fact-006',
 'fact-007',
 'fact-008']

## 1. Prewarm: load every model and run one pass per stage

This is the cost moved from the first user turn to startup.

In [2]:
prewarm = {}
t = time.monotonic()
tts = p.TTSWorker()
prewarm["kokoro_load_ms"] = tts.load_ms
prewarm["kokoro_warm_synth_ms"] = tts.prewarm()

asr = p.ASR()
prewarm["whisper_variant"] = asr.variant
prewarm["whisper_repo"] = asr.repo
prewarm["whisper_load_and_warm_ms"] = asr.prewarm()

ollama = p.OllamaLLM()
ollama.unload()                       # make the Ollama load an honest cold load
prewarm["ollama_load_and_warm_ms"] = ollama.load()

mlx = p.MlxLLM()
prewarm["mlx_load_and_warm_ms"] = mlx.load()
prewarm["total_ms"] = p.ms(t)
results["prewarm"] = prewarm
prewarm

<venv>/lib/python3.12/site-packages/torch/nn/modules/rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
<venv>/lib/python3.12/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


<venv>/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

{'kokoro_load_ms': 4423.0,
 'kokoro_warm_synth_ms': 1330.0,
 'whisper_variant': 'accent-lora',
 'whisper_repo': '<accent-lora>',
 'whisper_load_and_warm_ms': 12388.6,
 'ollama_load_and_warm_ms': 3932.2,
 'mlx_load_and_warm_ms': 3092.2,
 'total_ms': 25303.1}

## 2. Input audio

Kokoro speaks each question once. WAVs are cached in `audio/` and fed at 16 kHz.

In [3]:
audio = {i["id"]: p.input_audio(tts, i) for i in items}
{k: round(len(v) / 16000, 2) for k, v in audio.items()}

{'fact-001': 1.82,
 'fact-002': 1.71,
 'fact-003': 2.27,
 'fact-004': 1.81,
 'fact-005': 1.93,
 'fact-006': 2.0,
 'fact-007': 2.27,
 'fact-008': 1.81}

## 3. First turn after prewarm, per path

Reported separately, not part of the warm medians.

In [4]:
paths = {"ollama": ollama, "mlx": mlx}
turns = []
for name, llm in paths.items():
    turns += p.run_round(asr, llm, tts, items[:1], audio, phase="first", round_=0)
[{k: t[k] for k in ("path", "stt_ms", "llm_ttft_ms", "time_to_first_audio_ms", "reply_text")} for t in turns]

[{'path': 'ollama',
  'stt_ms': 509.6,
  'llm_ttft_ms': 830.0,
  'time_to_first_audio_ms': 2101.0,
  'reply_text': 'The capital of France is Paris.'},
 {'path': 'mlx',
  'stt_ms': 210.3,
  'llm_ttft_ms': 544.0,
  'time_to_first_audio_ms': 1198.4,
  'reply_text': 'The capital of France is Paris.'}]

## 4. Warm rounds

The path order alternates each round, so drift doesn't favour one path.

In [5]:
for r in range(1, ROUNDS + 1):
    order = ["ollama", "mlx"] if r % 2 else ["mlx", "ollama"]
    for name in order:
        turns += p.run_round(asr, paths[name], tts, items, audio, phase="warm", round_=r)
        print(f"round {r} {name}: done")
results["turns"] = turns
len(turns)

round 1 ollama: done


round 1 mlx: done


round 2 mlx: done


round 2 ollama: done


round 3 ollama: done


round 3 mlx: done


50

## 5. Memory (partial: this process plus Ollama's loaded model, not the whole voice room)

In [6]:
results["memory"] = {
    "python_max_rss_mb": round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20),
    **ollama.memory(), **mlx.memory(),
}
results["memory"]

{'python_max_rss_mb': 1606,
 'ollama_size_mb': 5572,
 'ollama_size_vram_mb': 5572,
 'mlx_active_mb': 2785,
 'mlx_peak_mb': 3492}

## 6. Summary and verdict

In [7]:
summary = {name: p.summarize(turns, name) for name in paths}
results["summary"] = summary
results["verdict"] = p.verdict(summary)
p.save(results)

cols = ["stt_ms", "llm_ttft_ms", "first_sentence_ms", "time_to_first_audio_ms",
        "llm_total_ms", "end_to_end_ms", "decode_tok_s"]
print(f"{'metric':<24}" + "".join(f"{n:>22}" for n in summary))
for c in cols:
    print(f"{c:<24}" + "".join(f"{s[c]:>13} ±{s[c + '_spread']:<7}" for s in summary.values()))
for n, s in summary.items():
    print(f"{n}: items correct {s['items_correct']}/{s['items_total']}, turns correct {s['turns_correct']}/{s['n']}")
results["verdict"]

metric                                  ollama                   mlx
stt_ms                          213.6 ±7.3            214.2 ±7.4    
llm_ttft_ms                     713.7 ±33.8           135.6 ±3.0    
first_sentence_ms              1465.0 ±50.1           651.0 ±11.2   
time_to_first_audio_ms         1872.4 ±499.5          860.4 ±12.8   
llm_total_ms                   1198.2 ±47.8           382.4 ±13.4   
end_to_end_ms                  1872.4 ±499.5          860.4 ±12.8   
decode_tok_s                     20.7 ±0.7             35.7 ±1.4    
ollama: items correct 8/8, turns correct 24/24
mlx: items correct 8/8, turns correct 24/24


{'tfa_budget_ms': 1500,
 'min_items_correct': 5,
 'per_path': {'ollama': {'time_to_first_audio_ms': 1872.4,
   'tfa_under_budget': False,
   'items_correct': '8/8',
   'correct_ok': True},
  'mlx': {'time_to_first_audio_ms': 860.4,
   'tfa_under_budget': True,
   'items_correct': '8/8',
   'correct_ok': True}},
 'claim_holds': False,
 'ttft_gap_ms': 578.1,
 'ttft_noise_ms': 33.8,
 'ttft_winner': 'mlx',
 'endpointing_ms_not_included': 1500}

## 7. Replies and wrong answers

In [8]:
for t in turns:
    if t["phase"] == "warm" and t["round"] == 1:
        flag = "" if t["correct"] else "  <-- WRONG"
        print(f"{t['path']:<7} {t['id']}  {t['transcript']!r} -> {t['reply_text']!r}{flag}")
print("\nwrong, all warm rounds:", [(t["path"], t["round"], t["id"], t["reply_text"]) for t in turns if t["phase"] == "warm" and not t["correct"]])
print("mlx cache hits (round 1):", [t.get("cache_hit_tokens") for t in turns if t["path"] == "mlx" and t["round"] == 1])
print("ollama prompt_eval_ms (round 1):", [t.get("prompt_eval_ms") for t in turns if t["path"] == "ollama" and t["round"] == 1])

ollama  fact-001  'What is the capital of France?' -> 'The capital of France is Paris.'
ollama  fact-002  'How many continents are there?' -> 'There are seven continents.'
ollama  fact-003  'What color is the sky on a clear day?' -> 'The sky is blue on a clear day.'
ollama  fact-004  'who wrote Romeo and Juliet.' -> 'William Shakespeare wrote Romeo and Juliet.'
ollama  fact-005  'How many days are in a leap year?' -> 'A leap year has 366 days.'
ollama  fact-006  'What is the largest ocean on Earth?' -> 'The largest ocean on Earth is the Pacific Ocean.'
ollama  fact-007  'What gas do plants absorb from the air?' -> 'Plants absorb carbon dioxide from the air.'
ollama  fact-008  'In which country is Mount Fuji?' -> 'Mount Fuji is in Japan.'
mlx     fact-001  'What is the capital of France?' -> 'The capital of France is Paris.'
mlx     fact-002  'How many continents are there?' -> 'There are seven continents.'
mlx     fact-003  'What color is the sky on a clear day?' -> 'The sky is blue on

## 8. Check: recompute the summary from results.json on disk

In [9]:
saved = json.loads(p.RESULTS_PATH.read_text())
for name in paths:
    assert p.summarize(saved["turns"], name) == saved["summary"][name], name
assert len([t for t in saved["turns"] if t["phase"] == "warm"]) == len(items) * ROUNDS * len(paths)
"results.json consistent" 

'results.json consistent'

## 9. Listen

Replies from warm round 1.

In [10]:
from IPython.display import Audio, display
for t in turns:
    if t["phase"] == "warm" and t["round"] == 1 and t["id"] in ("fact-001", "fact-004"):
        print(t["path"], t["id"], t["reply_text"])
        display(Audio(str(p.HERE / t["reply_wav"])))

ollama fact-001 The capital of France is Paris.


ollama fact-004 William Shakespeare wrote Romeo and Juliet.


mlx fact-001 The capital of France is Paris.


mlx fact-004 William Shakespeare wrote Romeo and Juliet.
